Nicheformer implementation:  
We want to test the cabability of Nicheformer in capturing the spatial niche information


the tokenized parquet file should be run previously with the command as:  
python /scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/benchmark/nicheformer/run_tokenize.py  

afterwards, there are batchs of parquet data generated in the ./data directory

In [2]:
import scanpy as sc
import anndata as ad
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import numba
import math
from tqdm import tqdm
from scipy.sparse import issparse
from sklearn.utils import sparsefuncs
import pyarrow.parquet as pq
import pyarrow
import os
import torch
import pandas as pd
import sys
sys.path.append("/scratch/project_465001027/nicheformer/src/nicheformer/models")
from _nicheformer import Nicheformer

In [72]:

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Building the input batch and dataloader

In [69]:
# Define the base directory
base_dir = "/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/benchmark/nicheformer/data"

# List all files in the directory
all_files = os.listdir(base_dir)

# Make sure to include the base directory in the file paths
full_paths = [os.path.join(base_dir, file) for file in all_files]

tables = []
# Print the full paths
for path in full_paths:
    tables.append(pd.read_parquet(path))
    
def generate_batches(tables, batch_size = 10, context_length=1500):
    for chunk in tables[:1]:  # Process only the first table
        for batch_num in range((len(chunk) + batch_size - 1) // batch_size):  # Correct range calculation for batches
            batch_df = chunk.iloc[batch_num * batch_size: (batch_num + 1) * batch_size]
            
            # Check if batch_df is empty
            if batch_df.empty:
                continue
            # Create the combined batch list
            batch_list = batch_df.apply(lambda x: list(x[["assay", "specie", "modality"]]) + list(x["X"]), axis=1)
            # Convert the batch_list into a tensor
            input_tensor = torch.tensor(batch_list.tolist(), dtype=torch.long)
            
            # Create the batch dictionary with input tensor
            batch = {'X': input_tensor[:,:context_length]}
            yield batch

In [75]:
# Step 3: Load the pre-trained Nicheformer model
def load_nicheformer_model(checkpoint_path, config):
    model = Nicheformer(**config)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    model.to(device)
    model.eval()
    return model

# Step 4: Extract embeddings using the model
def get_cell_embeddings(model, input_tensor_batches):
    embeddings_list = []
    with torch.no_grad():
        for batch in input_tensor_batches:
            input_tensor = batch['X']  # Assuming 'X' contains the tensor

            # Move the tensor to the appropriate device (GPU or CPU)
            input_tensor = input_tensor.to(device)

            # Create attention mask
            attention_mask = (input_tensor != 0)  # Padding token is assumed to be 1
            attention_mask = attention_mask.type(torch.bool).to(device)

            # Forward pass
            output = model(input_tensor, attention_mask=attention_mask)

            # Extract [CLS] token embeddings (assuming [CLS] is at the first position)
            cls_embeddings = output['transformer_output'][:, 0, :]
            embeddings_list.append(cls_embeddings.cpu())

    if not embeddings_list:
        print("Error: No embeddings were generated.")
        return None

    return torch.cat(embeddings_list, dim=0)

In [78]:
input_tensor_batches = generate_batches(tables, batch_size = 1, context_length=1500)

In [79]:
get_cell_embeddings(model, input_tensor_batches)

998


tensor([[ 0.2938, -0.0148, -0.0426,  ...,  1.1312,  0.9909, -1.3071],
        [ 0.4402, -0.1911, -0.2430,  ...,  0.8571,  0.8633, -1.5761],
        [ 0.4243, -0.1155, -0.0661,  ...,  1.1176,  0.9061, -1.4472],
        ...,
        [ 0.5579, -0.1056, -0.1666,  ...,  1.0056,  0.8699, -1.5564],
        [ 0.5021, -0.2592, -0.2806,  ...,  0.9913,  0.9057, -1.7589],
        [ 0.3701, -0.0898, -0.0293,  ...,  1.1156,  0.9864, -1.3087]])

In [8]:
# Model configuration
config = {
    'dim_model': 512,
    'nheads': 16,
    'dim_feedforward': 1024,
    'nlayers': 12,
    'dropout': 0.0,
    'batch_first': True,
    'masking_p': 0.15,
    'n_tokens': 20340,
    'context_length': 1500,
    'lr': 1e-4,
    'warmup': 1,
    'max_epochs': 50000,
    'pool': 'mean',
    'batch_size': 12,
    'autoregressive': False
}

checkpoint_path = '/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/benchmark/nicheformer/nicheformer.ckpt'

In [11]:
model = load_nicheformer_model(checkpoint_path, config)

In [13]:
model

Nicheformer(
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (linear1): Linear(in_features=512, out_features=1024, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (linear2): Linear(in_features=1024, out_features=512, bias=True)
    (norm1): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    (norm2): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    (dropout1): Dropout(p=0.0, inplace=False)
    (dropout2): Dropout(p=0.0, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear